## Lab 2
### **Part A- Email classification with Naive Bayes**

This notebook loads the supplied Lab2 dataset, creates a reproducible train/test split, trains GaussianNB and MultinomialNB, compares their held-out performance, and predicts labels for test emails. “Multinominal” is interpreted as Multinomial Naive Bayes.

Only email text is used as input. label_num is the target (0 = ham, 1 = spam); label and the unnamed row identifier are excluded from features to prevent target leakage.

In [46]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

RANDOM_STATE = 42
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)

Python: 3.14.7
pandas: 3.1.0 | NumPy: 2.5.4 | scikit-learn: 1.9.1


### 1. Load and inspect the dataset
Keeping the CSV in the root folder.

In [47]:
DATA_PATH = Path('Lab2_dataset.csv')
df = pd.read_csv(DATA_PATH)
print('Original shape:', df.shape)
display(df.head())
display(df.isna().sum().rename('missing_values').to_frame())
display(df['label'].value_counts().rename('emails').to_frame())
display(pd.crosstab(df['label'], df['label_num']))

Original shape: (5171, 4)


,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\nth...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\n( see a...",0
2,3624,ham,"Subject: neon retreat\nho ho ho , we ' re arou...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\nthis deal is to ...,0


,missing_values
Unnamed: 0,0
label,0
text,0
label_num,0


,emails
label,
ham,3672
spam,1499


label_num,0,1
label,,
ham,3672,0
spam,0,1499


### 2. Validate labels and remove repeated texts
The loaded dataset has no missing values. Exact duplicate texts are removed before splitting so the same email cannot occur in training and testing. Conflicting labels for identical texts raise an error. No stemming or stop-word removal is applied; the vectorizer uses lowercase word tokens.

In [48]:
assert {'text', 'label', 'label_num'}.issubset(df.columns)
assert df[['text', 'label', 'label_num']].notna().all().all()
assert df['text'].str.strip().ne('').all()
assert df['label'].map({'ham': 0, 'spam': 1}).eq(df['label_num']).all()
assert df.groupby('text')['label_num'].nunique().max() == 1, 'Conflicting labels'
duplicates = int(df['text'].duplicated().sum())
clean = df.drop_duplicates(subset='text').copy()
print(f'Removed {duplicates} duplicate texts; retained {len(clean)} emails.')
display(clean['label'].value_counts().rename('emails').to_frame())

Removed 178 duplicate texts; retained 4993 emails.


,emails
label,
ham,3531
spam,1462


### 3. Split into training and test sets
Using 80% of the data for training and 20% for testing, keeping a similar balance of spam and ham in both groups. A fixed random seed gives the same split each time

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    clean['text'], clean['label_num'], test_size=0.20,
    random_state=RANDOM_STATE, stratify=clean['label_num'])
assert set(X_train).isdisjoint(set(X_test))
print(f'Train: {len(X_train)} emails | Test: {len(X_test)} emails')
display(pd.DataFrame({'train': y_train.value_counts(), 'test': y_test.value_counts()})
        .rename(index={0: 'ham', 1: 'spam'}))

Train: 3994 emails | Test: 999 emails


,train,test
label_num,,
ham,2825,706
spam,1169,293


### 4. Convert text into word counts

Build a list of up to 2,000 words using only the training emails. Use this list to count words in both the training and test emails. Give both models the same word counts.
MultinomialNB works well with word counts. GaussianNB expects numbers that follow a bell-shaped pattern, so it may be less suitable. We include it to compare the results.
GaussianNB stores every count, including zeros. MultinomialNB can save space by storing only nonzero counts. The 2,000-word limit helps keep memory use manageable.

In [50]:
vectorizer = CountVectorizer(max_features=2000)
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)
X_train_dense = X_train_counts.toarray().astype(np.float64)
X_test_dense = X_test_counts.toarray().astype(np.float64)
print('Training feature matrix:', X_train_counts.shape)
print('Test feature matrix:', X_test_counts.shape)
print(f'Dense arrays: {(X_train_dense.nbytes + X_test_dense.nbytes) / 1024**2:.1f} MiB')

Training feature matrix: (3994, 2000)
Test feature matrix: (999, 2000)
Dense arrays: 76.2 MiB


### 5. Train both classifiers
Use fixed parameters without tuning on the test set. Additive smoothing for MultinomialNB is alpha=1.0; GaussianNB uses var_smoothing=1e-9.

In [51]:
models = {
    'GaussianNB': GaussianNB(var_smoothing=1e-9),
    'MultinomialNB': MultinomialNB(alpha=1.0),
}
models['GaussianNB'].fit(X_train_dense, y_train)
models['MultinomialNB'].fit(X_train_counts, y_train)
predictions = {
    'GaussianNB': models['GaussianNB'].predict(X_test_dense),
    'MultinomialNB': models['MultinomialNB'].predict(X_test_counts),
}
print('Both models trained successfully.')

Both models trained successfully.


### 6. Compare held-out performance

Primary comparison: accuracy. Spam precision measures how many spam flags are correct; spam recall measures how many spam emails are caught. Spam F1 balances precision and recall. A majority-class baseline provides context. Confusion matrices use rows = actual and columns = predicted.

In [52]:
rows = []
for name, pred in predictions.items():
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred),
                 'spam_precision': precision_score(y_test, pred, zero_division=0),
                 'spam_recall': recall_score(y_test, pred, zero_division=0),
                 'spam_f1': f1_score(y_test, pred, zero_division=0)})
results = pd.DataFrame(rows).set_index('model')
display(results.apply(lambda column: column.map(lambda value: f'{value:.2%}')))
majority = int(y_train.mode().iloc[0])
print(f'Majority-class test accuracy: {accuracy_score(y_test, np.full(len(y_test), majority)):.2%}')
for name, pred in predictions.items():
    print('\n' + name)
    print(classification_report(y_test, pred, labels=[0, 1], target_names=['ham', 'spam'], digits=4, zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
                         index=['actual ham', 'actual spam'],
                         columns=['predicted ham', 'predicted spam']))
results.to_csv('model_comparison.csv')

,accuracy,spam_precision,spam_recall,spam_f1
model,,,,
GaussianNB,94.69%,87.04%,96.25%,91.41%
MultinomialNB,94.19%,85.93%,95.90%,90.65%


Majority-class test accuracy: 70.67%

GaussianNB
              precision    recall  f1-score   support

         ham     0.9837    0.9405    0.9616       706
        spam     0.8704    0.9625    0.9141       293

    accuracy                         0.9469       999
   macro avg     0.9270    0.9515    0.9379       999
weighted avg     0.9505    0.9469    0.9477       999



,predicted ham,predicted spam
actual ham,664,42
actual spam,11,282



MultinomialNB
              precision    recall  f1-score   support

         ham     0.9821    0.9348    0.9579       706
        spam     0.8593    0.9590    0.9065       293

    accuracy                         0.9419       999
   macro avg     0.9207    0.9469    0.9322       999
weighted avg     0.9461    0.9419    0.9428       999



,predicted ham,predicted spam
actual ham,660,46
actual spam,12,281


In [53]:
best_accuracy = results['accuracy'].max()
winners = results.index[np.isclose(results['accuracy'], best_accuracy)].tolist()
best_name = winners[0]
if len(winners) > 1:
    print(f'The models tie on test accuracy ({best_accuracy:.2%}). GaussianNB is used for the example predictions by fixed order.')
else:
    other_name = next(name for name in models if name != best_name)
    gap = (best_accuracy - results.loc[other_name, 'accuracy']) * 100
    print(f'{best_name} performs better on this test split: {best_accuracy:.2%} accuracy, a {gap:.2f} percentage-point advantage.')
print('This is a descriptive comparison on one split, not proof of superiority on every dataset.')

GaussianNB performs better on this test split: 94.69% accuracy, a 0.50 percentage-point advantage.
This is a descriptive comparison on one split, not proof of superiority on every dataset.


### 7. Predict labels for test emails

Use the model with the highest test accuracy to predict whether test emails are spam or ham. Show the first 10 test emails, including any incorrect predictions.
The spam probability is the model’s estimate, not a guarantee. Save both models’ predictions for all test emails in a CSV file.

In [54]:
best_model = models[best_name]
best_input = X_test_dense if best_name == 'GaussianNB' else X_test_counts
best_predictions = best_model.predict(best_input)
spam_column = list(best_model.classes_).index(1)
spam_probabilities = best_model.predict_proba(best_input)[:, spam_column]
label_names = {0: 'ham', 1: 'spam'}
test_predictions = pd.DataFrame({
    'source_index': X_test.index,
    'text_preview': X_test.str.replace(r'\s+', ' ', regex=True).str.slice(0, 160).to_numpy(),
    'actual_label': y_test.map(label_names).to_numpy(),
    'gaussian_prediction': pd.Series(predictions['GaussianNB']).map(label_names).to_numpy(),
    'multinomial_prediction': pd.Series(predictions['MultinomialNB']).map(label_names).to_numpy(),
    'selected_model': best_name,
    'predicted_label': pd.Series(best_predictions).map(label_names).to_numpy(),
    'estimated_spam_probability': spam_probabilities,
    'correct': best_predictions == y_test.to_numpy(),
})
display(test_predictions.head(10))
test_predictions.to_csv('test_predictions.csv', index=False)
assert len(test_predictions) == len(y_test)
print(f'Saved {len(test_predictions)} test predictions.')

,source_index,text_preview,actual_label,gaussian_prediction,multinomial_prediction,selected_model,predicted_label,estimated_spam_probability,correct
0,3031,"Subject: frontrea plant , gas when : wednesday...",ham,spam,ham,GaussianNB,spam,1.0,False
1,4921,Subject: news at least two bids seen for enron...,ham,ham,ham,GaussianNB,ham,0.0,True
2,2262,"Subject: quailty web solutions , involutory ov...",spam,spam,spam,GaussianNB,spam,1.0,True
3,1438,Subject: txu fuels / sds nomination for decemb...,ham,ham,ham,GaussianNB,ham,0.0,True
4,676,Subject: use the links to get more information...,spam,spam,spam,GaussianNB,spam,1.0,True
5,4233,Subject: paliourg get the doctors time 4 freee...,spam,spam,spam,GaussianNB,spam,1.0,True
6,26,Subject: registration confirmation from spinne...,ham,spam,spam,GaussianNB,spam,1.0,False
7,4956,Subject: re : resume attached liz is an mit gr...,ham,ham,ham,GaussianNB,ham,0.0,True
8,2750,Subject: meter 981594 for jan the meter shows ...,ham,ham,ham,GaussianNB,ham,0.0,True
9,1571,"Subject: the investor ' s toolbox mcintyre , i...",spam,spam,spam,GaussianNB,spam,1.0,True


Saved 999 test predictions.


### **Part A- Interpretation and limitations**

The results show which model performed better on this train/test split. MultinomialNB usually suits word counts, but GaussianNB performed slightly better here.
Results may change with a different split, word limit, or model settings. We removed identical emails, but similar emails may remain.

We used the test results to choose the better model. Testing it on another unseen dataset would give a fairer assessment. 

## Part B: Airbnb New York City — price outliers

Load AB_NYC_2019.csv, compare common ways to find unusual nightly prices, and save a cleaned dataset.

We treat price as the listed nightly price in US dollars. It is not the total booking cost or a current market estimate. Each row is a listing; Entire home/apt identifies homes/apartments, while other rows are private or shared rooms. We keep all room types and compare similar listings.

An outlier is a value outside a chosen rule. A high price is not automatically an error. The aim is a dataset with no prices outside our fixed cleaning bounds, not a claim that every remaining listing is correct.

In [55]:


AIRBNB_PATH = Path('AB_NYC_2019.csv')
airbnb = pd.read_csv(AIRBNB_PATH)
print('Original dataset:', airbnb.shape)
display(airbnb.head())
display(airbnb.isna().sum().rename('missing').to_frame())
display(airbnb['price'].describe(percentiles=[.01, .05, .25, .50, .75, .95, .99]).to_frame())

Original dataset: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


,missing
id,0
name,16
host_id,0
host_name,21
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


,price
count,48895.000000
mean,152.720687
std,240.154170
min,0.000000
1%,30.000000
5%,40.000000
25%,69.000000
50%,106.000000
75%,175.000000
95%,355.000000


### 1. Separate invalid prices from statistical outliers

A zero or negative nightly price is not useful for this price analysis. Missing or nonnumeric prices would also be removed. These are data-validity checks, separate from outlier rules. We preserve the original columns and leave unrelated missing names and review fields unchanged.

Check duplicate listing IDs rather than deleting listings just because they have the same price. This file has unique IDs.

In [56]:
required = ['id', 'price', 'neighbourhood_group', 'room_type']
assert set(required).issubset(airbnb.columns)
assert airbnb['id'].notna().all() and airbnb['id'].is_unique
assert airbnb[['neighbourhood_group', 'room_type']].notna().all().all()
numeric_price = pd.to_numeric(airbnb['price'], errors='coerce')
valid_price = numeric_price.notna() & np.isfinite(numeric_price) & numeric_price.gt(0)
valid = airbnb.loc[valid_price].copy()
valid['price'] = numeric_price.loc[valid_price]
invalid = airbnb.loc[~valid_price].copy()
print(f'Invalid/nonpositive prices: {len(invalid):,}')
print(f'Positive-price listings used to calculate bounds: {len(valid):,}')
display(valid.groupby('room_type')['price'].agg(['count', 'mean', 'median', 'min', 'max']).round(2))
print(f"Mean nightly price: ${valid['price'].mean():.2f}; median: ${valid['price'].median():.2f}")

Invalid/nonpositive prices: 11
Positive-price listings used to calculate bounds: 48,884


,count,mean,median,min,max
room_type,,,,,
Entire home/apt,25407,211.81,160.0,10,10000
Private room,22319,89.81,70.0,10,10000
Shared room,1158,70.25,45.0,10,1800


Mean nightly price: $152.76; median: $106.00


### 2. Compare common outlier rules

The mean and median describe the middle of the data. Neither one is an outlier rule by itself: we need a measure of spread or a chosen cutoff too.


| Method | Prices kept | What it demonstrates |
|---|---|---|
| Mean ± 3 standard deviations | Within three standard deviations of the mean | Simple, but extreme prices raise both the mean and the cutoff; skewed prices do not follow a bell-shaped pattern. |
| Median / MAD | Modified absolute z-score ≤ 3.5 | Uses the median and median absolute deviation (MAD), so very expensive listings have less effect on the cutoff. |
| 1st–99th percentiles | Between the 1st and 99th percentiles, inclusive | Easy to explain, but cuts the tails by design, even when prices are valid. Ties mean removal is not exactly 2%. |
| Global IQR | Q1 − 1.5 × IQR through Q3 + 1.5 × IQR | Uses the middle 50% of prices; one citywide range can treat different room types unfairly. |
| Log-price IQR | Apply the IQR rule to log(price), then convert back | Allows a wider upper range for strongly right-skewed prices. |
| Borough + room-type IQR | IQR range among comparable listings | Accounts for location and room type; selected for the main cleaned dataset. |


IQR means Q3 − Q1. MAD means median(|price − median(price)|). For median/MAD, the modified z-score is 0.67448975 × (price − median) / MAD. All bounds below are learned once from the original positive-price rows. We do not repeatedly trim and recalculate until more listings disappear.

In [57]:
price = valid['price'].astype(float)
mean, sd = price.mean(), price.std(ddof=1)
median = price.median()
mad = (price - median).abs().median()
assert mad > 0
q1, q3 = price.quantile([.25, .75])
iqr = q3 - q1
log_price = np.log(price)
log_q1, log_q3 = log_price.quantile([.25, .75])
log_iqr = log_q3 - log_q1

global_bounds = {
    'Mean +/- 3 SD': (mean - 3 * sd, mean + 3 * sd),
    'Median/MAD (z=3.5)': (median - 3.5 * mad / .67448975, median + 3.5 * mad / .67448975),
    'Percentiles 1-99': tuple(price.quantile([.01, .99])),
    'Global IQR (1.5)': (q1 - 1.5 * iqr, q3 + 1.5 * iqr),
    'Log-price IQR (1.5)': (np.exp(log_q1 - 1.5 * log_iqr), np.exp(log_q3 + 1.5 * log_iqr)),
}
keep_masks = {}
bound_rows = []
for method, (lower, upper) in global_bounds.items():
    keep_masks[method] = price.between(lower, upper, inclusive='both')
    bound_rows.append({'method': method, 'lower_bound': lower, 'upper_bound': upper})
global_bounds_table = pd.DataFrame(bound_rows).set_index('method')
display(global_bounds_table.round(2))
print('Negative lower bounds do not allow invalid prices: those were already removed.')

,lower_bound,upper_bound
method,,
Mean +/- 3 SD,-567.76,873.27
Median/MAD (z=3.5),-132.70,344.70
Percentiles 1-99,30.00,799.00
Global IQR (1.5),-90.00,334.00
Log-price IQR (1.5),17.08,706.84


Negative lower bounds do not allow invalid prices: those were already removed.


### 3. Compare similar listings for the final rule

Group listings by borough and room type, then calculate a price range for each group using the IQR rule. For example, compare entire apartments in Manhattan with other entire apartments in Manhattan.

If a group has fewer than 30 listings, use the price range for that room type across all of NYC. This applies to shared rooms in Staten Island. The bounds are saved so the cleaning decision is reproducible.

In [58]:
GROUP_COLUMNS = ['neighbourhood_group', 'room_type']
MIN_GROUP_SIZE = 30
SELECTED_METHOD = 'Borough + room-type IQR (1.5)'
def iqr_summary(frame, keys):
    grouped = frame.groupby(keys)['price']
    stats = grouped.agg(count='size')
    stats['q1'] = grouped.quantile(.25)
    stats['q3'] = grouped.quantile(.75)
    stats['iqr'] = stats['q3'] - stats['q1']
    stats['lower_bound'] = stats['q1'] - 1.5 * stats['iqr']
    stats['upper_bound'] = stats['q3'] + 1.5 * stats['iqr']
    return stats

group_bounds = iqr_summary(valid, GROUP_COLUMNS)
room_bounds = iqr_summary(valid, 'room_type')
group_bounds['bounds_source'] = 'borough + room type'
for group_key in group_bounds.index:
    if group_bounds.loc[group_key, 'count'] < MIN_GROUP_SIZE:
        room_type = group_key[1]
        for col in ['q1', 'q3', 'iqr', 'lower_bound', 'upper_bound']:
            group_bounds.loc[group_key, col] = room_bounds.loc[room_type, col]
        group_bounds.loc[group_key, 'bounds_source'] = 'citywide room type (small group fallback)'

# Merge bounds by keys while keeping the original listing row index.
row_bounds = valid[GROUP_COLUMNS].join(group_bounds[['lower_bound', 'upper_bound']], on=GROUP_COLUMNS)
assert row_bounds[['lower_bound', 'upper_bound']].notna().all().all()
keep_masks[SELECTED_METHOD] = price.ge(row_bounds['lower_bound']) & price.le(row_bounds['upper_bound'])
display(group_bounds.round(2))
group_bounds.to_csv('airbnb_group_price_bounds.csv')

count     q1      q3     iqr  \
neighbourhood_group room_type                                       
Bronx               Entire home/apt    379   80.0  140.00   60.00   
                    Private room       651   40.0   70.00   30.00   
                    Shared room         60   28.0   55.50   27.50   
Brooklyn            Entire home/apt   9558  105.0  199.00   94.00   
                    Private room     10126   50.0   80.00   30.00   
                    Shared room        411   30.0   50.00   20.00   
Manhattan           Entire home/apt  13198  143.0  259.00  116.00   
                    Private room      7982   68.0  120.00   52.00   
                    Shared room        480   49.0   89.25   40.25   
Queens              Entire home/apt   2096   90.0  168.25   78.25   
                    Private room      3372   47.0   75.00   28.00   
                    Shared room        198   30.0   54.25   24.25   
Staten Island       Entire home/apt    176   75.0  151.25   76.25   
                    Private room       188   40.0   75.00   35.00   
                    Shared room          9   33.0   75.00   42.00   

                                     lower_bound  upper_bound  \
neighbourhood_group room_type                                   
Bronx               Entire home/apt       -10.00       230.00   
                    Private room           -5.00       115.00   
                    Shared room           -13.25        96.75   
Brooklyn            Entire home/apt       -36.00       340.00   
                    Private room            5.00       125.00   
                    Shared room             0.00        80.00   
Manhattan           Entire home/apt       -31.00       433.00   
                    Private room          -10.00       198.00   
                    Shared room           -11.38       149.62   
Queens              Entire home/apt       -27.38       285.62   
                    Private room            5.00       117.00   
                    Shared room            -6.38        90.62   
Staten Island       Entire home/apt       -39.38       265.62   
                    Private room          -12.50       127.50   
                    Shared room           -30.00       138.00   

                                                                 bounds_source  
neighbourhood_group room_type                                                   
Bronx               Entire home/apt                        borough + room type  
                    Private room                           borough + room type  
                    Shared room                            borough + room type  
Brooklyn            Entire home/apt                        borough + room type  
                    Private room                           borough + room type  
                    Shared room                            borough + room type  
Manhattan           Entire home/apt                        borough + room type  
                    Private room                           borough + room type  
                    Shared room                            borough + room type  
Queens              Entire home/apt                        borough + room type  
                    Private room                           borough + room type  
                    Shared room                            borough + room type  
Staten Island       Entire home/apt                        borough + room type  
                    Private room                           borough + room type  
                    Shared room      citywide room type (small group fallback)

### 4. Show how the choice changes the result

Every method starts from the same positive-price dataset, so the comparison is fair. “Below” and “above” count statistical outliers; the invalid prices removed earlier are not included in those counts.

More removals do not automatically mean better cleaning. A method can remove valid budget or luxury listings. The selected grouped IQR rule aims to compare similar listings without assuming prices follow a bell curve.

In [59]:
comparison_rows = []
for method, keep in keep_masks.items():
    if method == SELECTED_METHOD:
        below = price.lt(row_bounds['lower_bound'])
        above = price.gt(row_bounds['upper_bound'])
    else:
        lower, upper = global_bounds[method]
        below, above = price.lt(lower), price.gt(upper)
    retained_prices = price.loc[keep]
    comparison_rows.append({
        'method': method, 'input_positive_rows': len(valid),
        'below_bound': int(below.sum()), 'above_bound': int(above.sum()),
        'removed_outliers': int((~keep).sum()), 'kept_rows': int(keep.sum()),
        'removed_percent': float((~keep).mean() * 100),
        'kept_min': retained_prices.min(), 'kept_max': retained_prices.max(),
        'kept_mean': retained_prices.mean(), 'kept_median': retained_prices.median(),
    })
method_comparison = pd.DataFrame(comparison_rows).set_index('method')
display(method_comparison.round(2))
method_comparison.to_csv('airbnb_outlier_comparison.csv')

# Every flag identifies a row to remove under that method.
method_flags = pd.DataFrame({method: ~mask for method, mask in keep_masks.items()})
print('How often the methods disagree (positive-price listings):')
display(method_flags.sum(axis=1).value_counts().sort_index()
        .rename_axis('number_of_methods_flagging').rename('listings').to_frame())

# Review examples where local context changes the global IQR decision.
disagree = keep_masks['Global IQR (1.5)'].ne(keep_masks[SELECTED_METHOD])
examples = valid.loc[disagree, ['id', 'neighbourhood_group', 'room_type', 'price']].copy()
examples['kept_by_global_iqr'] = keep_masks['Global IQR (1.5)'].loc[disagree]
examples['kept_by_grouped_iqr'] = keep_masks[SELECTED_METHOD].loc[disagree]
examples = examples.join(row_bounds[['lower_bound', 'upper_bound']])
display(examples.sort_values('price', ascending=False).head(12))

,input_positive_rows,below_bound,above_bound,removed_outliers,kept_rows,removed_percent,kept_min,kept_max,kept_mean,kept_median
method,,,,,,,,,,
Mean +/- 3 SD,48884,0,388,388,48496,0.79,10.0,860.0,138.78,105.0
Median/MAD (z=3.5),48884,0,2916,2916,45968,5.97,10.0,344.0,120.27,100.0
Percentiles 1-99,48884,393,474,867,48017,1.77,30.0,799.0,138.50,105.0
Global IQR (1.5),48884,0,2972,2972,45912,6.08,10.0,334.0,120.00,100.0
Log-price IQR (1.5),48884,37,589,626,48258,1.28,18.0,700.0,136.20,105.0
Borough + room-type IQR (1.5),48884,0,3477,3477,45407,7.11,10.0,433.0,122.56,100.0


How often the methods disagree (positive-price listings):


,listings
number_of_methods_flagging,
0,44155
1,1769
2,854
3,1517
4,115
5,86
6,388


,id,neighbourhood_group,room_type,price,kept_by_global_iqr,kept_by_grouped_iqr,lower_bound,upper_bound
36657,29138655,Manhattan,Entire home/apt,433,False,True,-31.0,433.0
1435,644833,Manhattan,Entire home/apt,433,False,True,-31.0,433.0
41988,32634720,Manhattan,Entire home/apt,432,False,True,-31.0,433.0
39546,30792566,Manhattan,Entire home/apt,432,False,True,-31.0,433.0
41414,32237634,Manhattan,Entire home/apt,431,False,True,-31.0,433.0
38826,30398902,Manhattan,Entire home/apt,430,False,True,-31.0,433.0
35516,28187115,Manhattan,Entire home/apt,430,False,True,-31.0,433.0
48626,36349092,Manhattan,Entire home/apt,430,False,True,-31.0,433.0
4241,2791688,Manhattan,Entire home/apt,430,False,True,-31.0,433.0
47097,35596923,Manhattan,Entire home/apt,429,False,True,-31.0,433.0


### 5. Another option: percentile filtering

Keep listings priced between the 1st and 99th percentiles, including the cutoffs. Calculate these cutoffs from the original positive-price listings. Here the range is $30 to $799 per night.

Remove listings below $30 or above $799. For example, remove a $15 listing and a $1,000 listing, but keep a $150 listing at its original price. 
This removes 867 listings and keeps 48,017 listings. All retained prices stay unchanged. Show the price summary before and after filtering, check the retained prices now saved as a separate dataset B_NYC_2019_percentile_filtered.csv.



In [60]:
percentile_low, percentile_high = global_bounds['Percentiles 1-99']
percentile_keep = price.between(percentile_low, percentile_high, inclusive='both')
airbnb_percentile = airbnb.loc[valid.index[percentile_keep]].copy()
print(f'Percentile range: ${percentile_low:.2f} to ${percentile_high:.2f}')
print(f'Removed {(~percentile_keep).sum():,} positive-price listings outside the range.')
print(f'Kept {len(airbnb_percentile):,} listings; retained prices stay unchanged.')
display(pd.DataFrame({'original_positive_prices': price.describe(),
                      'after_percentile_filter': airbnb_percentile['price'].describe()}).round(2))
assert percentile_keep.equals(keep_masks['Percentiles 1-99'])
assert airbnb_percentile['price'].between(percentile_low, percentile_high).all()
assert airbnb_percentile.equals(airbnb.loc[airbnb_percentile.index])
assert len(airbnb_percentile) + int((~percentile_keep).sum()) == len(valid)
airbnb_percentile.to_csv('AB_NYC_2019_percentile_filtered.csv', index=False)

Percentile range: $30.00 to $799.00
Removed 867 positive-price listings outside the range.
Kept 48,017 listings; retained prices stay unchanged.


,original_positive_prices,after_percentile_filter
count,48884.00,48017.00
mean,152.76,138.50
std,240.17,103.69
min,10.00,30.00
25%,69.00,70.00
50%,106.00,105.00
75%,175.00,175.00
max,10000.00,799.00


### 6. Clean prices using the median and MAD

Calculate the median price for each borough and room type. Then calculate MAD: the median distance of prices from that group's median.

Keep prices with a modified z-score of 3.5 or less in absolute value: 0.67448975 × |price − median| / MAD ≤ 3.5. This gives the range median ± (3.5 × MAD / 0.67448975). Remove listings outside that range and save the result as AB_NYC_2019_clean.csv. Retained prices stay unchanged.

Groups with fewer than 30 listings use citywide median/MAD bounds for their room type. The median gives the typical price; MAD measures the spread needed to decide which prices are unusually far away. The median alone is not an outlier rule.
The earlier alternative is AB_NYC_2019_percentile_filtered.csv, which keeps prices from $30 to $799 across the whole city. The two outputs use different rules on the same starting data. This method starts again from the positive-price rows

The audit file gives one row per original listing, its removal reason, applicable group bounds, and outlier flags for all six methods. Statistical flags are left blank for invalid prices because those rows were excluded before bounds were calculated. 

In [61]:
selected_keep = keep_masks[SELECTED_METHOD]
airbnb_clean = airbnb.loc[valid.index[selected_keep]].copy()
airbnb_removed = airbnb.loc[~airbnb.index.isin(airbnb_clean.index)].copy()

audit = airbnb[['id', 'neighbourhood_group', 'room_type', 'price']].copy()
audit['valid_positive_price'] = valid_price
audit['lower_bound'] = row_bounds['lower_bound'].reindex(airbnb.index)
audit['upper_bound'] = row_bounds['upper_bound'].reindex(airbnb.index)
audit['decision'] = 'kept'
audit.loc[~valid_price, 'decision'] = 'invalid_or_nonpositive_price'
audit.loc[valid.index[price.lt(row_bounds['lower_bound'])], 'decision'] = 'below_group_median_mad_bound'
audit.loc[valid.index[price.gt(row_bounds['upper_bound'])], 'decision'] = 'above_group_median_mad_bound'
for method, mask in keep_masks.items():
    audit['outlier: ' + method] = (~mask).astype('boolean').reindex(airbnb.index)

# Verify using the original saved bounds, not new bounds fitted after trimming.
clean_bounds = row_bounds.loc[airbnb_clean.index]
assert airbnb_clean['price'].gt(0).all()
assert airbnb_clean['price'].ge(clean_bounds['lower_bound']).all()
assert airbnb_clean['price'].le(clean_bounds['upper_bound']).all()
assert airbnb_clean.equals(airbnb.loc[airbnb_clean.index])
assert len(airbnb_clean) + len(airbnb_removed) == len(airbnb)
assert audit['decision'].eq('kept').sum() == len(airbnb_clean)
assert all(row['kept_rows'] + row['removed_outliers'] == len(valid) for row in comparison_rows)

airbnb_clean.to_csv('AB_NYC_2019_clean.csv', index=False)
airbnb_removed.to_csv('AB_NYC_2019_removed.csv', index=False)
audit.to_csv('airbnb_price_audit.csv', index=False)
reloaded = pd.read_csv('AB_NYC_2019_clean.csv')
assert reloaded['id'].tolist() == airbnb_clean['id'].tolist()
assert np.array_equal(reloaded['price'].to_numpy(), airbnb_clean['price'].to_numpy())

print(f'Original listings: {len(airbnb):,}')
print(f'Invalid prices removed: {len(invalid):,}')
print(f'Grouped median/MAD outliers removed: {(~selected_keep).sum():,}')
print(f'Final cleaned listings: {len(airbnb_clean):,}')
print('Remaining outliers under the saved cleaning rule: 0')
display(pd.DataFrame({'before_positive_prices': price.describe(),
                      'after_grouped_median_mad': airbnb_clean['price'].describe()}).round(2))
print('Retention by borough and room type:')
retention = valid.groupby(GROUP_COLUMNS).size().rename('before').to_frame()
retention['after'] = airbnb_clean.groupby(GROUP_COLUMNS).size().reindex(retention.index, fill_value=0)
retention['removed'] = retention['before'] - retention['after']
retention['retained_percent'] = 100 * retention['after'] / retention['before']
display(retention.round(2))
retention.to_csv('airbnb_group_retention.csv')

Original listings: 48,895
Invalid prices removed: 11
Grouped median/MAD outliers removed: 3,477
Final cleaned listings: 45,407
Remaining outliers under the saved cleaning rule: 0


,before_positive_prices,after_grouped_median_mad
count,48884.00,45407.00
mean,152.76,122.56
std,240.17,75.26
min,10.00,10.00
25%,69.00,65.00
50%,106.00,100.00
75%,175.00,160.00
max,10000.00,433.00


Retention by borough and room type:


before  after  removed  retained_percent
neighbourhood_group room_type                                                
Bronx               Entire home/apt     379    346       33             91.29
                    Private room        651    612       39             94.01
                    Shared room          60     55        5             91.67
Brooklyn            Entire home/apt    9558   8946      612             93.60
                    Private room      10126   9528      598             94.09
                    Shared room         411    366       45             89.05
Manhattan           Entire home/apt   13198  12147     1051             92.04
                    Private room       7982   7324      658             91.76
                    Shared room         480    432       48             90.00
Queens              Entire home/apt    2096   1974      122             94.18
                    Private room       3372   3152      220             93.48
                    Shared room         198    177       21             89.39
Staten Island       Entire home/apt     176    159       17             90.34
                    Private room        188    181        7             96.28
                    Shared room           9      8        1             88.89

### **Part B: Interpretation and limitations**

- Mean-based rules are simple, but a few very expensive listings can make the allowed range too wide.
- Median/MAD and IQR are less affected by extreme prices. A citywide cutoff still mixes different markets.
- Percentile filtering gives a clear citywide range: $30–$799 here. It removes 867 positive-price listings and keeps 48,017, with their original prices. Valid low and high prices can be removed by design.
- Grouped median/MAD is our final choice because it compares similar room types in the same borough. It is a transparent choice, not a proven best rule. Boroughs still contain different neighbourhoods, and we do not have detailed amenities or property size to judge every listing.
- The cleaned dataset has no outliers under the fixed saved rule. Other rules may still flag retained prices. Recalculating medians and MAD after filtering can also give different bounds; we do not keep deleting rows until every possible rule agrees.
- Valid budget and luxury listings may be removed, which can change the population represented. Use the audit and removed-listings files to review decisions. No manual labels are available to measure outlier-detection accuracy.
- This is cleaning of nightly prices only, using a historical 2019 dataset. Missing review fields remain; they are not automatically errors. These prices do not describe the current NYC rental market.
Files to use: AB_NYC_2019_clean.csv for the main grouped-median/MAD dataset; AB_NYC_2019_percentile_filtered.csv for the percenntile filtering alternative; airbnb_outlier_comparison.csv for the method comparison; airbnb_group_price_bounds.csv and airbnb_price_audit.csv to explain each decision.